In [1]:
print("Notebook working successfully")

Notebook working successfully


In [2]:
import pandas as pd
import numpy as np
import os

print("Libraries loaded successfully")

Libraries loaded successfully


In [3]:
orders = pd.read_csv('../dataset/olist_orders_dataset.csv')
customers = pd.read_csv('../dataset/olist_customers_dataset.csv')
items = pd.read_csv('../dataset/olist_order_items_dataset.csv')
products = pd.read_csv('../dataset/olist_products_dataset.csv')
sellers = pd.read_csv('../dataset/olist_sellers_dataset.csv')
payments = pd.read_csv('../dataset/olist_order_payments_dataset.csv')
reviews = pd.read_csv('../dataset/olist_order_reviews_dataset.csv')
geo = pd.read_csv('../dataset/olist_geolocation_dataset.csv')
category_translation = pd.read_csv('../dataset/product_category_name_translation.csv')

print("All tables loaded successfully")

All tables loaded successfully


In [4]:
for name, df in [
    ('orders', orders),
    ('customers', customers),
    ('items', items),
    ('products', products)
]:
    print(f"\n========== {name.upper()} ==========")
    print(f"Shape: {df.shape}")
    print("\nNull Values:")
    print(df.isnull().sum())
    print("\nData Types:")
    print(df.dtypes)


========== ORDERS ==========
Shape: (99441, 8)

Null Values:
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

Data Types:
order_id                         object
customer_id                      object
order_status                     object
order_purchase_timestamp         object
order_approved_at                object
order_delivered_carrier_date     object
order_delivered_customer_date    object
order_estimated_delivery_date    object
dtype: object

========== CUSTOMERS ==========
Shape: (99441, 5)

Null Values:
customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

Data Types:
customer_id                 objec

In [5]:
date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_cols:
    orders[col] = pd.to_datetime(orders[col], errors='coerce')

print("Date columns converted successfully")
print("\nUpdated Data Types:")
print(orders[date_cols].dtypes)

Date columns converted successfully

Updated Data Types:
order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object


In [6]:
# Delivery days: purchase → customer delivery
orders['delivery_days'] = (
    orders['order_delivered_customer_date'] -
    orders['order_purchase_timestamp']
).dt.days

# Late delivery flag
orders['is_late'] = (
    orders['order_delivered_customer_date'] >
    orders['order_estimated_delivery_date']
).astype(int)

# Month and Year extraction
orders['order_month'] = orders['order_purchase_timestamp'].dt.to_period('M')
orders['order_year'] = orders['order_purchase_timestamp'].dt.year

print("New columns created successfully")

# Preview
print(
    orders[
        ['delivery_days', 'is_late', 'order_month', 'order_year']
    ].head()
)

New columns created successfully
   delivery_days  is_late order_month  order_year
0            8.0        0     2017-10        2017
1           13.0        0     2018-07        2018
2            9.0        0     2018-08        2018
3           13.0        0     2017-11        2017
4            2.0        0     2018-02        2018


In [7]:
# Merge product table with English category translation
products = products.merge(
    category_translation,
    on='product_category_name',
    how='left'
)

# Fill missing category names
products['product_category_name_english'] = products[
    'product_category_name_english'
].fillna('Unknown')

print("Products table cleaned successfully")

# Preview
print(
    products[
        ['product_category_name',
         'product_category_name_english']
    ].head()
)

Products table cleaned successfully
   product_category_name product_category_name_english
0             perfumaria                     perfumery
1                  artes                           art
2          esporte_lazer                sports_leisure
3                  bebes                          baby
4  utilidades_domesticas                    housewares


In [8]:
# Merge orders + customers
master = orders.merge(
    customers[['customer_id', 'customer_state', 'customer_city']],
    on='customer_id',
    how='left'
)

# Merge order items
master = master.merge(
    items[['order_id', 'product_id', 'seller_id',
           'price', 'freight_value']],
    on='order_id',
    how='left'
)

# Merge products
master = master.merge(
    products[['product_id',
              'product_category_name_english']],
    on='product_id',
    how='left'
)

# Merge payments
master = master.merge(
    payments[['order_id',
              'payment_type',
              'payment_value']],
    on='order_id',
    how='left'
)

# Merge reviews
master = master.merge(
    reviews[['order_id',
             'review_score']],
    on='order_id',
    how='left'
)

print("Master dataset created successfully")
print("Shape:", master.shape)

master.head()

Master dataset created successfully
Shape: (119143, 22)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_days,is_late,...,customer_state,customer_city,product_id,seller_id,price,freight_value,product_category_name_english,payment_type,payment_value,review_score
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,8.0,0,...,SP,sao paulo,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,29.99,8.72,housewares,credit_card,18.12,4.0
1,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,8.0,0,...,SP,sao paulo,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,29.99,8.72,housewares,voucher,2.00,4.0
2,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,8.0,0,...,SP,sao paulo,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,29.99,8.72,housewares,voucher,18.59,4.0
3,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,13.0,0,...,BA,barreiras,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962,118.70,22.76,perfumery,boleto,141.46,4.0
4,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,9.0,0,...,GO,vianopolis,aa4383b373c6aca5d8797843e5594415,4869f7a5dfa277a7dca6462dcf3b52b2,159.90,19.22,auto,credit_card,179.12,5.0


In [9]:
# Save cleaned files
master.to_csv('../cleaned-data/master_orders.csv', index=False)
orders.to_csv('../cleaned-data/orders_clean.csv', index=False)
products.to_csv('../cleaned-data/products_clean.csv', index=False)

print("All cleaned files saved successfully")

All cleaned files saved successfully


In [10]:
print("Master Dataset Shape:", master.shape)

print("\nColumns:")
print(master.columns.tolist())

print("\nNull Values in Key Columns:")
print(
    master[
        ['order_id',
         'customer_id',
         'payment_value',
         'review_score']
    ].isnull().sum()
)

Master Dataset Shape: (119143, 22)

Columns:
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'delivery_days', 'is_late', 'order_month', 'order_year', 'customer_state', 'customer_city', 'product_id', 'seller_id', 'price', 'freight_value', 'product_category_name_english', 'payment_type', 'payment_value', 'review_score']

Null Values in Key Columns:
order_id           0
customer_id        0
payment_value      3
review_score     997
dtype: int64
